In [42]:
!pip install -q -U google-genai sentence-transformers chromadb langchain langchain-core langchain-text-splitters langgraph langchain-google-genai

In [43]:
import os
import json
import re
import pandas as pd

from google.colab import userdata
from sentence_transformers import SentenceTransformer
import chromadb

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

print("All libraries imported successfully.")

All libraries imported successfully.


In [44]:
# Load API key from Google Colab Secrets

api_key = None

try:
    api_key = userdata.get("NeyKey")
except Exception:
    pass

if not api_key:
    api_key = os.environ.get("GOOGLE_API_KEY")

if not api_key:
    raise ValueError("Please add geminiApiKey to Colab Secrets.")

os.environ["GOOGLE_API_KEY"] = api_key

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
)

print("Gemini connection ready!")

Gemini connection ready!


In [50]:
test_response = llm.invoke(
    "Reply with exactly: Gemini connection works."
)

print(test_response.content)

[{'type': 'text', 'text': 'Gemini connection works.', 'extras': {'signature': 'EpQECpEEARFNMg84AwR+WQp6Uil4WQpda73uUbI7GfAHf0tocKuwPmmm3ilkcRJqw0i+rwFnqTuBVTCQUHk6UbvBxPKgzx+Qq0h/P7OThYYl3ztiAomXxezfVMUetKvIz68OIcLs0UryT77HGH3awQuhXfP0hyjnLYnM95uSR9BjJFEWCKr1E5w5a2MirbexNbCbpMyze4dRUrqyl0RviJvBhbH/FAnJR/LMHjl4AgSZH6xo64ce+gALfY9J1BZuYgD+RQCkJuO/5ymjqZxkOWaWvkbgXazMrpnRfsPWOAMEcGaR6AzgsXuc6pyPVJERnnaVNDLXmVrW+3NVRSw4MMID17NIArmo7r+ufLC4UjINxyTgoz8CSPxmdqRhLa1VbLwngrAzaXGLDZ/4Lpz2r9oWGFjm8bL8/vwPhJAXZNE1O8uPNc744YarOjGXPvMCelsZj4fOCK+E2ByikFIXs16BJGRjno3C3pnY4SRpA/V35mkt6Y6OXKmlz3N+/TISPf5r1HkJ8obYe7u8ZM+kNzAbnhK16EYBQXVIsVGs9Idn3ipUyNaWIz7RH58Nyoc0HlGtWEDqdvQJrvtBsFFx6BS5dUzVYBMxEGXj/2Wv+tQvCiaPJplAlEGKZcp+1HcFL+AZxSikVj36RSaIBMZcQjynFWh1odtzp7sg1jBPpWRItGzWXxQTSpY0i77knPTbkfKYuX20xQ=='}}]


In [51]:
policies = [

    Document(
        page_content=(
            "Visitor Policy: All visitors must report to the school security desk, "
            "show valid identification, state the purpose of the visit, and receive "
            "a visitor pass before entering restricted areas."
        ),
        metadata={"source": "Visitor Policy"}
    ),

    Document(
        page_content=(
            "Unauthorized Access Policy: Any person attempting to enter a restricted "
            "school area without permission should be stopped when safe to do so. "
            "Security staff should verify identity and inform the school administration."
        ),
        metadata={"source": "Unauthorized Access Policy"}
    ),

    Document(
        page_content=(
            "Suspicious Activity Policy: Suspicious behavior, unattended objects, "
            "repeated attempts to bypass security, or unusual activity near sensitive "
            "areas should be documented and reported to the security supervisor."
        ),
        metadata={"source": "Suspicious Activity Policy"}
    ),

    Document(
        page_content=(
            "Emergency Policy: During serious threats, security staff should prioritize "
            "student and staff safety, notify school administration and emergency "
            "services when appropriate, and follow the school emergency response procedure."
        ),
        metadata={"source": "Emergency Policy"}
    ),

    Document(
        page_content=(
            "Incident Reporting Policy: Security incidents should record the date, "
            "time, location, people involved, observed activity, immediate action taken, "
            "and recommended follow-up. High-risk incidents should be escalated to the "
            "appropriate authority."
        ),
        metadata={"source": "Incident Reporting Policy"}
    )
]

print("Policy documents:", len(policies))

Policy documents: 5


In [52]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(policies)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(
        f"Chunk {i+1}: {chunk.metadata['source']}"
    )

Number of chunks: 5
Chunk 1: Visitor Policy
Chunk 2: Unauthorized Access Policy
Chunk 3: Suspicious Activity Policy
Chunk 4: Emergency Policy
Chunk 5: Incident Reporting Policy


In [53]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="school_security_policies"
)

texts = [
    doc.page_content
    for doc in chunks
]

metadatas = [
    doc.metadata
    for doc in chunks
]

embeddings = embedding_model.encode(
    texts
).tolist()

ids = [
    f"policy_{i}"
    for i in range(len(chunks))
]

collection.upsert(
    ids=ids,
    documents=texts,
    metadatas=metadatas,
    embeddings=embeddings
)

print("Policies stored in ChromaDB.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Policies stored in ChromaDB.


In [54]:
def retrieve_policies(query, k=3):

    query_embedding = embedding_model.encode(
        [query]
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k
    )

    retrieved = []

    for text, metadata in zip(
        results["documents"][0],
        results["metadatas"][0]
    ):

        retrieved.append({
            "policy": metadata["source"],
            "text": text
        })

    return retrieved

In [55]:
test_results = retrieve_policies(
    "A visitor entered a restricted area without permission."
)

for item in test_results:

    print("\n", item["policy"], ":")
    print(item["text"])


 Unauthorized Access Policy :
Unauthorized Access Policy: Any person attempting to enter a restricted school area without permission should be stopped when safe to do so. Security staff should verify identity and inform the school administration.

 Visitor Policy :
Visitor Policy: All visitors must report to the school security desk, show valid identification, state the purpose of the visit, and receive a visitor pass before entering restricted areas.

 Suspicious Activity Policy :
Suspicious Activity Policy: Suspicious behavior, unattended objects, repeated attempts to bypass security, or unusual activity near sensitive areas should be documented and reported to the security supervisor.


In [56]:
class SecurityAgentState(TypedDict, total=False):

    incident: str

    retrieved_policies: list

    severity: str

    summary: str

    recommended_action: str

    reason: str

    needs_escalation: bool

    final_response: str

In [57]:
def retrieve_node(state: SecurityAgentState):

    retrieved = retrieve_policies(
        state["incident"],
        k=3
    )

    return {
        "retrieved_policies": retrieved
    }

In [58]:
analysis_prompt = ChatPromptTemplate.from_template(
'''
You are a school security administration assistant.

Analyze the following incident using ONLY the supplied school security policies.

INCIDENT:
{incident}

RELEVANT POLICIES:
{policies}

Return ONLY valid JSON with these keys:
severity, summary, recommended_action, reason

severity must be exactly LOW, MEDIUM, or HIGH.

Do not invent policy rules that are not present in the supplied policies.
'''
)

analysis_chain = analysis_prompt | llm

In [59]:
def analysis_node(state: SecurityAgentState):

    policy_text = "\n\n".join(
        [
            f"{p['policy']}: {p['text']}"
            for p in state["retrieved_policies"]
        ]
    )

    response = analysis_chain.invoke({

        "incident": state["incident"],

        "policies": policy_text
    })

    # Get Gemini response content
    content = response.content

    # Gemini/LangChain may return content as a list
    if isinstance(content, list):

        text_parts = []

        for block in content:

            if isinstance(block, dict):

                text_parts.append(
                    block.get("text", "")
                )

            else:

                text_parts.append(
                    str(block)
                )

        text = "".join(
            text_parts
        ).strip()

    else:

        text = str(
            content
        ).strip()

    # Remove Markdown JSON code fences
    text = re.sub(
        r"^```json\s*",
        "",
        text
    )

    text = re.sub(
        r"^```\s*",
        "",
        text
    )

    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    # Convert Gemini JSON response into Python dictionary
    try:

        data = json.loads(text)

    except Exception:

        data = {

            "severity": "MEDIUM",

            "summary": text,

            "recommended_action":
                "Review the incident according to school security procedures.",

            "reason":
                "The model response could not be parsed as JSON."
        }

    return {

        "severity":
            data.get(
                "severity",
                "MEDIUM"
            ),

        "summary":
            data.get(
                "summary",
                ""
            ),

        "recommended_action":
            data.get(
                "recommended_action",
                ""
            ),

        "reason":
            data.get(
                "reason",
                ""
            )
    }

In [60]:
def escalation_node(state: SecurityAgentState):

    severity = state.get(
        "severity",
        "MEDIUM"
    ).upper()

    return {
        "needs_escalation":
            severity == "HIGH"
    }

In [61]:
def final_node(state: SecurityAgentState):

    escalation = (
        "YES"
        if state.get("needs_escalation")
        else "NO"
    )

    report = f"""
SCHOOL SECURITY INCIDENT REPORT

Incident:
{state['incident']}

Severity:
{state.get('severity', 'MEDIUM')}

Summary:
{state.get('summary', '')}

Recommended Action:
{state.get('recommended_action', '')}

Reason:
{state.get('reason', '')}

Escalation Required:
{escalation}

Relevant Policies:
"""

    for policy in state.get(
        "retrieved_policies",
        []
    ):

        report += (
            f"- {policy['policy']}\n"
        )

    return {
        "final_response": report
    }

In [62]:
graph_builder = StateGraph(
    SecurityAgentState
)

# Add nodes
graph_builder.add_node(
    "retrieve_policies",
    retrieve_node
)

graph_builder.add_node(
    "analyze_incident",
    analysis_node
)

graph_builder.add_node(
    "check_escalation",
    escalation_node
)

graph_builder.add_node(
    "final_report",
    final_node
)

# Workflow
graph_builder.add_edge(
    START,
    "retrieve_policies"
)

graph_builder.add_edge(
    "retrieve_policies",
    "analyze_incident"
)

graph_builder.add_edge(
    "analyze_incident",
    "check_escalation"
)

graph_builder.add_edge(
    "check_escalation",
    "final_report"
)

graph_builder.add_edge(
    "final_report",
    END
)

# Compile
security_agent = graph_builder.compile()

print(
    "LangGraph agent compiled successfully!"
)

LangGraph agent compiled successfully!


In [63]:
incident = """
A person who is not a registered visitor was seen trying to enter
a restricted laboratory area without permission. The security guard
stopped the person and informed the school administration.
"""

initial_state = {
    "incident": incident
}

result = security_agent.invoke(
    initial_state
)

print(
    result["final_response"]
)


SCHOOL SECURITY INCIDENT REPORT

Incident:

A person who is not a registered visitor was seen trying to enter
a restricted laboratory area without permission. The security guard
stopped the person and informed the school administration.


Severity:
MEDIUM

Summary:
An unregistered visitor attempted to enter a restricted laboratory area without permission and was intercepted by a security guard who notified school administration.

Recommended Action:
Verify the individual's identity as required by the Unauthorized Access Policy, ensure they comply with the Visitor Policy by registering at the security desk, and document and report the incident to the security supervisor per the Suspicious Activity Policy.

Reason:
The individual violated the Visitor Policy by failing to report to the security desk and obtain a pass, and engaged in unauthorized access by attempting to enter a restricted laboratory area. Attempting to enter a restricted/sensitive area also constitutes unusual activity un

In [64]:
print("--- AGENT STATE ---")

for key, value in result.items():

    if key != "final_response":

        print(f"\n{key}:")

        print(value)

--- AGENT STATE ---

incident:

A person who is not a registered visitor was seen trying to enter
a restricted laboratory area without permission. The security guard
stopped the person and informed the school administration.


retrieved_policies:
[{'policy': 'Unauthorized Access Policy', 'text': 'Unauthorized Access Policy: Any person attempting to enter a restricted school area without permission should be stopped when safe to do so. Security staff should verify identity and inform the school administration.'}, {'policy': 'Visitor Policy', 'text': 'Visitor Policy: All visitors must report to the school security desk, show valid identification, state the purpose of the visit, and receive a visitor pass before entering restricted areas.'}, {'policy': 'Suspicious Activity Policy', 'text': 'Suspicious Activity Policy: Suspicious behavior, unattended objects, repeated attempts to bypass security, or unusual activity near sensitive areas should be documented and reported to the security sup

In [65]:
incident = """
A student was found attempting to enter the computer laboratory
after school hours without authorization.
"""

initial_state = {
    "incident": incident
}

result = security_agent.invoke(
    initial_state
)

print(result["final_response"])


SCHOOL SECURITY INCIDENT REPORT

Incident:

A student was found attempting to enter the computer laboratory
after school hours without authorization.


Severity:
LOW

Summary:
A student attempted to enter the restricted computer laboratory after school hours without authorization.

Recommended Action:
Stop the student when safe to do so, verify their identity, inform the school administration, and document the event to report to the security supervisor.

Reason:
The incident violates the Unauthorized Access Policy, which requires stopping unauthorized individuals attempting to enter restricted areas, verifying their identity, and notifying administration. Furthermore, attempting entry to a sensitive area after hours constitutes unusual activity under the Suspicious Activity Policy, requiring documentation and reporting to the security supervisor.

Escalation Required:
NO

Relevant Policies:
- Unauthorized Access Policy
- Visitor Policy
- Suspicious Activity Policy



In [66]:
incident = """
An unattended bag was found near the main school entrance.
The security guard informed the security supervisor and school administration.
"""

initial_state = {
    "incident": incident
}

result = security_agent.invoke(
    initial_state
)

print(result["final_response"])


SCHOOL SECURITY INCIDENT REPORT

Incident:

An unattended bag was found near the main school entrance.
The security guard informed the security supervisor and school administration.


Severity:
LOW

Summary:
An unattended bag was discovered near the main school entrance, and the security guard reported it to the security supervisor and school administration.

Recommended Action:
Document the incident as required by policy.

Reason:
Under the Suspicious Activity Policy, unattended objects must be documented and reported to the security supervisor. The guard has already notified the supervisor and administration, so completing the formal documentation is the remaining required action.

Escalation Required:
NO

Relevant Policies:
- Emergency Policy
- Unauthorized Access Policy
- Suspicious Activity Policy

